<a href="https://colab.research.google.com/github/pandeynivedita7/codingal/blob/main/simplenumberperdcition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# MNIST step-by-step program with inline explanations (comments) and small runtime prints.
# This file runs in PyCharm or Colab. Keep it simple.

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# -------------------------
# STEP 0: (optional) seeds for repeatability
# -------------------------
np.random.seed(42)
tf.random.set_seed(42)

# -------------------------
# STEP 1: Load the MNIST dataset
# - x_train/x_test: arrays of shape (N, 28, 28), pixel values 0-255
# - y_train/y_test: integer labels 0-9
# -------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print("STEP 1: data shapes:", x_train.shape, y_train.shape, x_test.shape, y_test.shape)

# -------------------------
# STEP 2: Inspect a sample value range (before normalization)
# -------------------------
print("STEP 2: sample pixel range before normalize:", x_train.min(), x_train.max())

# -------------------------
# STEP 3: Normalize pixels from [0,255] -> [0.0,1.0]
# - convert to float32 then divide by 255
# -------------------------
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0
print("STEP 3: sample pixel range after normalize:", x_train.min(), x_train.max())

# -------------------------
# STEP 4: Build a simple Dense (fully-connected) model
# - Flatten: convert 28x28 -> 784 vector
# - Dense(128, relu): hidden layer with 128 neurons
# - Dense(10, softmax): output probabilities for 10 classes (0-9)
# -------------------------
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),     # flatten 2D -> 1D
    layers.Dense(128, activation='relu'),     # hidden layer
    layers.Dense(10, activation='softmax')    # output layer
])
print("STEP 4: model created. Summary below:")
model.summary()  # prints shape and parameter counts

# -------------------------
# STEP 5: Compile the model
# - optimizer: Adam (good default)
# - loss: sparse_categorical_crossentropy (labels are integer class indices)
# - metrics: accuracy
# -------------------------
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
print("STEP 5: model compiled with Adam optimizer and sparse_categorical_crossentropy loss.")

# -------------------------
# STEP 6: Train the model
# - epochs: number of passes over training data
# - batch_size: number of samples per weight update
# - validation_split: fraction of training set used for validation
# -------------------------
print("STEP 6: start training...")
history = model.fit(
    x_train, y_train,
    epochs=5,            # increase to 10-20 for better accuracy
    batch_size=64,
    validation_split=0.1 # 10% of training used for validation
)
print("STEP 6: training finished.")

# -------------------------
# STEP 7: Evaluate on the test set (unseen data)
# - returns test loss and test accuracy
# -------------------------
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"STEP 7: Test accuracy = {test_acc:.4f}, Test loss = {test_loss:.4f}")

# -------------------------
# STEP 8: Make predictions on test set
# - model.predict returns probability vectors (softmax outputs)
# - np.argmax picks the class with highest probability
# -------------------------
predictions = model.predict(x_test)          # shape (10000, 10)
print("STEP 8: predictions shape:", predictions.shape)
first_probs = predictions[0]
print("STEP 8: first sample probabilities (first 6 values):", first_probs[:6])
predicted_label = np.argmax(first_probs)
print("STEP 8: predicted label for first test image:", predicted_label, "  actual label:", y_test[0])

# -------------------------
# STEP 9: Visualize the first test image with labels
# -------------------------
plt.imshow(x_test[0], cmap='gray')
plt.title(f"Predicted: {predicted_label}   Actual: {y_test[0]}")
plt.axis('off')
plt.show()

# -------------------------
# EXTRA (optional): plot training/validation accuracy and loss curves
# -------------------------
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.show()
